In [1]:
import os
import sys
import urllib.request

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from stellargraph.layer import GCN_LSTM

In [2]:
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df['praia'] = df[['City', 'Beach']].apply(lambda x: ' - '.join(x), axis=1)
print(f'Numero de praias: {len(df.praia.unique())}')
df = df.sort_values(by=['praia'])
df = df.sort_values(by=['Date'])
df = df[['Date','praia','Enterococcus']]
df.shape

Numero de praias: 174


(69016, 3)

In [3]:
for praia in df.praia.unique():
    df_aux = df.loc[df['praia']==praia]
    #print(f'Praia: {praia} - Qtd Medições: {len(df_aux.Date)}')
    print(praia)

BERTIOGA - BORACÉIA - COL. MARISTA
SÃO VICENTE - ITARARÉ (POSTO 2)
UBATUBA - TENÓRIO
BERTIOGA - ENSEADA - INDAIÁ
MONGAGUÁ - AGENOR DE CAMPOS
CARAGUATATUBA - COCANHA
SÃO VICENTE - MILIONÁRIOS
SÃO VICENTE - PRAIA DA DIVISA
CARAGUATATUBA - MASSAGUAÇU (R. MARIA CARLOTA)
ITANHAÉM - SUARÃO
ITANHAÉM - SONHO
MONGAGUÁ - CENTRAL
ILHA ANCHIETA - PRAIA DE FORA
ITANHAÉM - PRAIA DOS PESCADORES
GUARUJÁ - ENSEADA (R. CHILE)
ITANHAÉM - PARQUE BALNEÁRIO
SÃO VICENTE - PRAINHA ( AV. SANTINO BRITO)
ITANHAÉM - JARDIM SÃO FERNANDO
UBATUBA - DOMINGAS DIAS
CARAGUATATUBA - PORTO NOVO
ITANHAÉM - JARDIM CIBRATEL
UBATUBA - DURA
ITANHAÉM - ESTÂNCIA BALNEÁRIA
SÃO VICENTE - PRAIA DA ILHA PORCHAT
MONGAGUÁ - ITAPOÃ - VILA SÃO PAULO
GUARUJÁ - ENSEADA (ESTR. DE PERNAMBUCO)
CARAGUATATUBA - PRAINHA
PRAIA GRANDE - AVIAÇÃO
IGUAPE - JURÉIA
SÃO SEBASTIÃO - SANTIAGO
PERUÍBE - PRAINHA
PERUÍBE - PERUÍBE (R. ICARAÍBA)
ILHA ANCHIETA - PRAIA DAS PALMAS
SÃO SEBASTIÃO - SAÍ
PERUÍBE - PERUÍBE (PARQUE TURÍSTICO)
CARAGUATATUBA - MOCOÓCA


In [4]:
df_aux=df.pivot(index='praia', columns='Date', values='Enterococcus' )
df_aux.fillna(df_aux.mean(), inplace=True)
df_aux=df_aux.astype('int32')
df_aux
#d = {'praia': [praia]}
#df_novo = pd.DataFrame(data=d) 
#pd.concat([df_novo, df_aux['Enterococcus']], axis=1)
#df_novo = df_novo.join(df_aux['Enterococcus'].T)
#df_novo

Date,2012-01-03,2012-01-08,2012-01-15,2012-01-22,2012-01-29,2012-02-05,2012-02-12,2012-02-19,2012-02-26,2012-03-04,...,2020-07-27,2020-08-03,2020-08-10,2020-08-17,2020-08-24,2020-08-31,2020-09-07,2020-09-14,2020-09-21,2020-09-28
praia,,,,,,,,,,,,,,,,,,,,,
BERTIOGA - BORACÉIA - COL. MARISTA,8,22,17,8,2,1,68,32,1,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - BORACÉIA - SUL,16,1,1,15,7,2,18,45,4,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - COLÔNIA DO SESC,128,42,18,31,11,19,252,36,1,2,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - INDAIÁ,17,85,1,10,108,6,7,29,15,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - R. RAFAEL COSTABILI,204,34,19,33,10,25,288,57,42,2,...,3,43,3,30,30,8,2,2,152,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UBATUBA - SUNUNGA,5,6,34,9,3,5,9,2,3,1,...,15,20,18,48,33,35,39,36,287,91
UBATUBA - TENÓRIO,9,96,42,14,7,17,5,1,12,5,...,9,1,3,14,2,1,12,1,86,4
UBATUBA - TONINHAS,1,128,21,11,13,2,13,21,28,2,...,36,9,3,7,9,18,10,2,188,41


In [16]:
test_size=20
adj_praias = np.zeros((174,174))
treino = df_aux.iloc[:,:-test_size].copy()
treino = treino.values.reshape((1,treino.shape[0],treino.shape[1]))
teste = df_aux.iloc[:,-test_size:].copy()
teste = teste.values.reshape((1,teste.shape[0],teste.shape[1]))
print(treino.shape)
print(teste.shape)
print(adj_praias.shape)

(1, 174, 409)
(1, 174, 20)
(174, 174)


In [21]:
gcn_lstm = GCN_LSTM(
    seq_len=treino.shape[2],
    adj=adj_praias,
    gc_layer_sizes=[16, 10],
    gc_activations=["relu", "relu"],
    lstm_layer_sizes=[200, 200],
    lstm_activations=["tanh", "tanh"],
)
x_input, x_output = gcn_lstm.in_out_tensors()
model = Model(inputs=x_input, outputs=x_output)
model.summary()

C:\Users\User\anaconda3\envs\StellarGraph-gpu\lib\site-packages\ipykernel_launcher.py:7: ExperimentalWarning: GCN_LSTM is experimental: Lack of unit tests and code refinement (see: https://github.com/stellargraph/stellargraph/issues/1132, https://github.com/stellargraph/stellargraph/issues/1526, https://github.com/stellargraph/stellargraph/issues/1564). It may be difficult to use and may have major changes at any time.
  import sys
C:\Users\User\anaconda3\envs\StellarGraph-gpu\lib\site-packages\stellargraph\core\utils.py:157: RuntimeWarning: divide by zero encountered in power
  D = np.diag(np.ravel(adj.sum(axis=0)) ** (-0.5))


Model: "model_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_7 (InputLayer)         [(None, 174, 409)]        0         
_________________________________________________________________
tf_op_layer_ExpandDims_3 (Te [(None, 174, 409, 1)]     0         
_________________________________________________________________
reshape_9 (Reshape)          (None, 174, 409)          0         
_________________________________________________________________
fixed_adjacency_graph_convol (None, 174, 16)           36994     
_________________________________________________________________
fixed_adjacency_graph_convol (None, 174, 10)           30610     
_________________________________________________________________
reshape_10 (Reshape)         (None, 174, None, 1)      0         
_________________________________________________________________
permute_3 (Permute)          (None, None, 174, 1)      0   

In [22]:
model.compile(optimizer="adam", loss="mae", metrics=["mse"])

In [23]:
history = model.fit(
    treino,
    teste,
    epochs=100,
    batch_size=60,
    shuffle=True,
    verbose=1,
    #validation_data=[testX, testY],
)

Train on 1 samples
Epoch 1/100
1/1 [==============================] - 1s 716ms/sample


ValueError: Dimensions must be equal, but are 174 and 20 for 'loss/dense_3_loss/sub' (op: 'Sub') with input shapes: [?,174], [?,174,20].